# Task C — step 1: train P_θ (21-dim path DDPM)

Pre-registered decisions: `taskc/DECISIONS.md` (read it before running; nothing here may change it).

Cell 0 is the Colab clone/checkout cell from `v2_experiments.ipynb`. **Locally**, skip it and start at cell 1 with the repo root as the working directory.

What this notebook does: build the Heston-P training set → global standardizer → train the MLP DDPM with the validated loop → save checkpoint → draw the three disjoint P_θ samples A/B/C with the frozen sampler → training-health diagnostics. It does **not** run the step-2 gate; that is `taskc/gate.py` and the next notebook.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, time, math
if os.path.basename(os.getcwd()) == "notebooks":      # local run from notebooks/
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch
import matplotlib.pyplot as plt

import taskc                                   # puts repo root + taskb/ on sys.path
from taskc.config import CFG
from taskc.data import build_training_set, make_loader
from taskc.ptheta import (make_schedule, build_model, train_ptheta,
                          save_checkpoint, load_checkpoint,
                          sample_ptheta, save_draw, load_draw)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

## Config

All values come from `taskc/config.py` (pre-registered). The only knobs here are run-control: whether to retrain or load the checkpoint, and whether to draw fewer paths for a CPU dry run (a reduced draw is **not** a valid input to the gate or the dual — regenerate at full size before step 2).

In [ ]:
from dataclasses import replace
RUN_TAG   = "rung0"         # artifacts go to artifacts_taskc/<RUN_TAG>/ ; one tag per rung of the ladder
RETRAIN   = True            # False -> load artifacts_taskc/<RUN_TAG>/ptheta_mlp21.pt
EPOCHS    = CFG.epochs      # rung 0: 150 ; rung 1/2: 450 (DECISIONS.md section 9)
SNR       = False           # rung 2 only: Min-SNR-gamma t-reweighting (Hang et al. 2023)
DRAW_N    = None            # None -> CFG.draws[*].n (1e5 each); e.g. 10_000 for a CPU dry run
DRAWS     = ("A", "B", "C")

RUN = replace(CFG, artifact_dir=CFG.run_dir(RUN_TAG), epochs=EPOCHS, snr_weighting=SNR)
CKPT = os.path.join(RUN.artifact_dir, RUN.ckpt_name)
os.makedirs(RUN.artifact_dir, exist_ok=True)

print(f"H={RUN.H} dt=1/{round(1/RUN.dt)} S0={RUN.S0} r={RUN.r}")
print(f"train: Heston-P seed {RUN.train_seed}, N={RUN.n_train:,}   ref (gate): seed {RUN.ref_seed}, N={RUN.n_ref:,}")
print(f"DDPM: T={RUN.T} cosine s={RUN.cosine_s} | MLP hidden={RUN.hidden_dim} time_emb={RUN.time_emb_dim} | "
      f"epochs={RUN.epochs} batch={RUN.batch_size} lr={RUN.lr} wd={RUN.weight_decay} | snr_weighting={RUN.snr_weighting} (gamma {RUN.snr_gamma})")
print(f"outlier rule: reject max|z| > {RUN.z_cap} | draws: " +
      ", ".join(f"{k}: seed {d.seed}, n {d.n:,}" for k, d in RUN.draws.items()))
print("artifacts ->", RUN.artifact_dir)

## Training set: Heston-P → global standardizer → cap

In [ ]:
t0 = time.time()
ts = build_training_set(RUN)
std = ts.std
print(f"built in {time.time()-t0:.1f}s")
print(f"standardizer: m={std.m:.6e}  s={std.s:.6e}   (global scalars, fitted on all N x H returns)")
print(f"training paths kept: {ts.z.shape[0]:,}  rejected by cap {CFG.z_cap}: {ts.n_rejected}   "
      f"max|z| in data = {ts.max_abs_z:.2f}")
print(f"per-column sd of Y: min {ts.Y_raw.std(0).min():.6f} max {ts.Y_raw.std(0).max():.6f}  (stationary start -> global std loses nothing)")
assert ts.n_rejected == 0, "cap binds on real data -- stop, see DECISIONS.md section 4"

sched = make_schedule(RUN, device=device)
print(f"schedule: T={sched.T}  abar[0]={float(sched.alphas_bar[0]):.6f}  abar[T-1]={float(sched.alphas_bar[-1]):.2e}  "
      f"| frozen sampler starts at t_start={sched.t_start} (beta_{sched.t_start}={float(sched.betas[sched.t_start]):.3f}, "
      f"skipped beta_{sched.T-1}={float(sched.betas[-1]):.3f}, abar_{sched.t_start}={float(sched.alphas_bar[sched.t_start]):.1e})")

## Train (published loop, reused unchanged) or load

Reused because it is the same code and architecture as the paper — not because its published outputs were validated (see `docs/04_MQ_DESTANDARDIZATION_AUDIT.md`).

In [ ]:
if RETRAIN:
    model = build_model(RUN).to(device)
    loader = make_loader(ts.z, batch_size=RUN.batch_size, seed=RUN.init_seed)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"params: {n_params:,}   steps/epoch: {len(loader)}   total steps: {len(loader)*EPOCHS:,}")
    t0 = time.time()
    model = train_ptheta(model, loader, sched, RUN, device=device)
    train_seconds = time.time() - t0
    print(f"trained in {train_seconds/60:.1f} min")
    save_checkpoint(CKPT, model, std, RUN,
                    extra=dict(tag=RUN_TAG, epochs=EPOCHS, snr_weighting=SNR, train_seconds=train_seconds, device=str(device),
                               n_train_kept=int(ts.z.shape[0]), max_abs_z_train=ts.max_abs_z))
    print("saved", CKPT)
else:
    model, std_ck, cfg_dict, extra = load_checkpoint(CKPT, device=device)
    assert std_ck == std, "checkpoint standardizer != training-set standardizer"
    print("loaded", CKPT, extra)
model.eval();

## Post-training ε-MSE by diffusion time (training health, not the gate)

For standardized Gaussian-like data the optimal ε-MSE is ≈ ᾱ_t (≈1 near t=0 where x_t carries no information about ε, ≈0 near t=T where x_t ≈ ε), so the informative quantity is the **excess** MSE − ᾱ_t, which should be ≤ 0 everywhere once the net has learned the non-Gaussian structure. The second table is the relative error of the near-identity map ε̂ ≈ x_t at the top of the chain: the sampler starts at `t_start` (998), where this error is amplified by 1/√α_t; it is the reason the clipped step 999 is skipped (DECISIONS.md §3).

In [ ]:
@torch.no_grad()
def eps_mse_by_t(model, z, sched, n_bins=10, n=20_000, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(z.shape[0], generator=g)[:n]
    x0 = torch.from_numpy(z[idx.numpy()]).to(device)
    T = sched.T
    rows = []
    for b in range(n_bins):
        lo, hi = int(b * T / n_bins), int((b + 1) * T / n_bins)
        t = torch.randint(lo, hi, (x0.shape[0],), generator=g).to(device)
        ab = sched.alphas_bar[t].view(-1, 1)
        noise = torch.randn(x0.shape, generator=g).to(device)
        xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
        pred = model(xt, (t.float() + 0.5) / T)
        rows.append((lo, hi, ((pred - noise) ** 2).mean().item(), ab.mean().item()))
    return rows

print(f"{'t range':>12s} {'eps-MSE':>8s} {'abar':>7s} {'excess':>8s}")
rows = eps_mse_by_t(model, ts.z, sched)
for lo, hi, m, ab in rows:
    print(f"[{lo:4d},{hi:4d}) {m:8.4f} {ab:7.4f} {m-ab:+8.4f}")

@torch.no_grad()
def identity_map_error(model, sched, ts_list, B=4000, seed=0):
    torch.manual_seed(seed)
    y = torch.randn(B, CFG.data_dim, device=device)
    out = []
    for t in ts_list:
        e = model(y, torch.full((B,), (t + 0.5) / sched.T, device=device))
        ref = torch.sqrt(1 - sched.alphas_bar[t]) * y
        rel = ((e - ref) ** 2).mean().sqrt() / ref.pow(2).mean().sqrt()
        out.append((t, rel.item(), float(sched.betas[t]), float(1 / torch.sqrt(sched.alphas[t]))))
    return out

print(f"\n{'t':>4s} {'rel.err eps_hat vs x_t':>24s} {'beta_t':>7s} {'1/sqrt(alpha_t)':>16s}")
for t, rel, b, amp in identity_map_error(model, sched, [999, 998, 997, 995, 990, 950]):
    flag = "  <- skipped (clipped beta)" if t > sched.t_start else ("  <- t_start" if t == sched.t_start else "")
    print(f"{t:4d} {rel:24.4f} {b:7.4f} {amp:16.3f}{flag}")

xs = [ (lo+hi)/2 for lo,hi,_,_ in rows ]
plt.figure(figsize=(6,3)); plt.plot(xs, [m-ab for _,_,m,ab in rows], marker="o"); plt.axhline(0, c="k", lw=.8)
plt.xlabel("t"); plt.ylabel("eps-MSE - abar_t"); plt.grid(alpha=.3); plt.title("excess eps-MSE over the Gaussian optimum"); plt.show()

## The three disjoint P_θ draws (frozen ancestral sampler from t_start = 998 down to 0, cap applied)

Seeds are fixed in `taskc/config.py`. Rejection counts are part of the record; anything above a handful of paths per 1e5 is a training problem, not a cap problem.

In [ ]:
draws = {}
for name in DRAWS:
    d = RUN.draws[name]
    n = d.n if DRAW_N is None else int(DRAW_N)
    print(f"--- draw {name}: seed {d.seed}, n={n:,}")
    res = sample_ptheta(model, sched, n=n, seed=d.seed, cfg=RUN, device=device, verbose=True)
    save_draw(RUN, name, res)
    draws[name] = res
    print(f"    done: {res.n_drawn:,} drawn, {res.n_rejected} rejected ({res.reject_rate*100:.4f}%), "
          f"max|z|={np.abs(res.z).max():.2f}, {res.seconds/60:.1f} min")
if DRAW_N is not None:
    print("\n*** reduced draws: NOT valid for the gate or the dual; rerun with DRAW_N=None ***")

## Training-health diagnostics (not the gate)

Column-wise mean/sd of z for draw A vs the training set, marginal histogram, and the per-path max|z| tail. The gate (`taskc/gate.py`) compares against an *independent* Heston-P sample with the pre-registered thresholds; these plots only tell you whether training obviously failed.

In [ ]:
zA = draws["A"].z if "A" in draws else load_draw(RUN, "A").z
zT = ts.z
print(f"{'col':>3s} {'mean_train':>11s} {'mean_A':>9s} {'sd_train':>9s} {'sd_A':>7s}")
for j in range(RUN.H):
    print(f"{j+1:3d} {zT[:,j].mean():11.4f} {zA[:,j].mean():9.4f} {zT[:,j].std():9.4f} {zA[:,j].std():7.4f}")
print(f"\nglobal: mean train {zT.mean():+.4f}  A {zA.mean():+.4f} | sd train {zT.std():.4f}  A {zA.std():.4f}")
print(f"MC floor for a column mean at N={zA.shape[0]:,}: {1/np.sqrt(zA.shape[0]):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
bins = np.linspace(-6, 6, 121)
ax[0].hist(zT.ravel(), bins=bins, density=True, alpha=.5, label="train"); ax[0].hist(zA.ravel(), bins=bins, density=True, alpha=.5, label="P_theta A")
ax[0].set_yscale("log"); ax[0].set_title("marginal of z (log density)"); ax[0].legend()
pmT, pmA = np.abs(zT).max(1), np.abs(zA).max(1)
q = np.linspace(0.9, 1.0, 200)
ax[1].plot(q, np.quantile(pmT, q), label="train"); ax[1].plot(q, np.quantile(pmA, q), label="P_theta A"); ax[1].axhline(RUN.z_cap, ls="--", c="k", lw=1, label="cap")
ax[1].set_title("per-path max|z| quantiles"); ax[1].legend()
ax[2].plot(zT.mean(0), marker="o", label="train"); ax[2].plot(zA.mean(0), marker="x", label="P_theta A"); ax[2].set_title("column means of z"); ax[2].legend()
plt.tight_layout(); plt.show()

# a few sample price paths, de-standardized once at the output
p = std.to_paths(zA[:20])
plt.figure(figsize=(6,3)); plt.plot(p.S.T, lw=.8); plt.title("20 P_theta price paths (S0 = %g)" % p.S0); plt.show()

## Stop here.

Step 1 deliverables produced by this notebook: `artifacts_taskc/<RUN_TAG>/ptheta_mlp21.pt`, `artifacts_taskc/<RUN_TAG>/ptheta_draw_{A,B,C}.npz`.

Next: `taskc/gate.py` + `notebooks/taskc_02_gate.ipynb` (same `RUN_TAG`) evaluate draw A against the independent Heston-P reference with the thresholds in `DECISIONS.md` §7. Do not build constraints or solve the dual on these draws before the gate passes. The same run, end to end without plots, is `python -m taskc.run_rung --tag <RUN_TAG> --epochs <E> [--snr]`.